In [1]:
from src.flow_match_dit.flowMatch import FlowMatch
from src.flow_match_dit.dit import DiT
from src.data.mnist import get_mnist_loader_and_transform
from src.data.cifar10 import get_cifar10_loader_and_transform
from torchvision.utils import save_image, make_grid
from src.flow_match_dit.train import train
import matplotlib.pyplot as plt
import torch
import glob
import os

In [2]:
T = 1000
dataset = "cifar10" # can be "cifar10" or "mnist"

PATH_TO_READY_MODEL = None # input path for ready model

PATH_TO_SAVE_MODEL = f"flow_model_dit_{dataset}.pth"
EPOCHS = 100 # for cifar10 it should be more than 1000, but for mnist 20-100 should be okay

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
if dataset == "mnist":
    data = get_mnist_loader_and_transform(batch_size=32)
elif dataset == "cifar10":
    data = get_cifar10_loader_and_transform(batch_size=32)

In [12]:
flow = FlowMatch(
    T = T,
    p_cond=0.9,
    time_shift=1,
    eps_model=DiT(
        in_channels=data.in_channels,
        out_channels=data.out_channels,
        T=T+1,
        num_classes=data.num_classes,
        steps=data.recommended_steps,
        num_encoder=12,
        hidden_size=768,
        n_heads=12,
        patch_size=4
    ),
    device=device
)
flow.to(device)
print(flow.device)


cuda


In [13]:


def latest_ckpt(ckpt_dir="checkpoints", prefix="ckpt_step", suffix=".pt"):
    if not os.path.isdir(ckpt_dir):
        return None
    files = glob.glob(os.path.join(ckpt_dir, f"{prefix}*{suffix}"))
    if not files:
        return None
    files.sort(key=lambda f: int(os.path.basename(f)[len(prefix):-len(suffix)]))
    return files[-1]

In [15]:
import os
# ---------- 1. optimizer 先建好 ----------
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-4)
start_step = 0

# ---------- 2. 选一个 ckpt 加载(可选) ----------
ckpt_to_load = "checkpoints"
ckpt_to_load = latest_ckpt("checkpoints")   # 返回具体文件路径, 或 None
print("will load:", ckpt_to_load)

if ckpt_to_load is not None and os.path.exists(ckpt_to_load):
    ckpt = torch.load(ckpt_to_load, map_location=device)
    if isinstance(ckpt, dict) and "model" in ckpt:
        # 完整 ckpt:一起恢复 optimizer / step
        flow.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_step = ckpt.get("step", 0)
        print(f"✅ 从 {ckpt_to_load} 恢复, step={start_step}")
    else:
        # 裸 state_dict:只能恢复模型权重
        flow.load_state_dict(ckpt)
        print(f"✅ 从 {ckpt_to_load} 加载模型权重(无 optimizer/step)")
else:
    print("🆕 从头开始训练")

# ---------- 3. 训练 ----------
save_dir =  "checkpoints"

train_losses, val_losses = train(
    model=flow,
    optimizer=optimizer,            # ← 用同一个 optimizer,不是新建的
    epochs=EPOCHS,
    device=device,
    train_loader=data.train_loader,
    val_loader=data.val_loader,
    save_dir=save_dir,
    save_interval=50,
    start_step=start_step,          # ← train() 新增这个参数
)

# ---------- 4. 画曲线 ----------
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train Loss")
if len(val_losses) > 0:
    plt.plot(val_losses, label="Val Loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "loss_curve.png"), dpi=150)
plt.show()

will load: checkpoints\ckpt_step00020400.pt
✅ 从 checkpoints\ckpt_step00020400.pt 恢复, step=20400


epoch 0:   0%|          | 1/1563 [01:50<47:56:21, 110.49s/it]


KeyboardInterrupt: 

In [ ]:
n_samples = 10
classes = torch.full((n_samples,), 2, dtype=torch.long)
x_t = flow.sample(n_samples=n_samples, size=data.train_dataset[0][0].shape,classes=classes,cond_weight=0.8)

result = []
for i in range(x_t.shape[0]):
    result.append(data.transform_to_pil(x_t[i]))

grid = make_grid(x_t, nrow=10)
save_image(grid, f"sample.png")

cols = 5
rows = (n_samples // cols) + (0 if n_samples % cols == 0 else 1)
fig, axs = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
for i in range(len(result)):
    row = i // cols
    axs[row, i % cols].imshow(result[i], cmap='gray')